# Tiny ImageNet ViT (PyTorch + timm)

This notebook implements a custom ~1 Million parameter Vision Transformer for Tiny ImageNet (64x64).

**Architecture:**
* **Library:** `timm` (PyTorch Image Models)
* **Input:** 64x64 images
* **Patch Size:** 8
* **Dim:** 96 (Embedding dimension)
* **Depth:** 8 layers
* **Heads:** 4
* **Params:** ~600k

In [1]:

import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import timm
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import requests
import zipfile
import io
import torchinfo
from torchinfo import summary
from PIL import Image

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Data Preparation
Tiny ImageNet validation folder needs formatting to work with PyTorch `ImageFolder`.

In [2]:
DATA_DIR = './tiny-imagenet-200'

def download_and_format_tiny_imagenet(root_dir):
    # 1. Download if not exists
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip" #alternative to duwe's link, given by CS231n and reccommended by projects
    if not os.path.exists(root_dir):
        print("Downloading Tiny ImageNet...")
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        z.extractall('./')
        print("Extracted.")
    else:
        print("Dataset already exists.")

    # 2. Format Validation Set (Move images into class folders)
    val_dir = os.path.join(root_dir, 'val')
    val_img_dir = os.path.join(val_dir, 'images')
    val_annotations = os.path.join(val_dir, 'val_annotations.txt')
    
    # Check if already formatted
    if os.path.exists(val_img_dir) and len(os.listdir(val_img_dir)) > 0:
        print("Formatting validation set...")
        with open(val_annotations, 'r') as f:
            for line in f:
                parts = line.split('\t')
                img_name = parts[0]
                class_label = parts[1]
                
                # Create class folder if not exists
                class_folder = os.path.join(val_dir, class_label)
                if not os.path.exists(class_folder):
                    os.makedirs(class_folder)
                
                # Move image
                src = os.path.join(val_img_dir, img_name)
                dst = os.path.join(class_folder, img_name)
                if os.path.exists(src):
                    os.rename(src, dst)
        
        # Cleanup empty images folder
        if not os.listdir(val_img_dir):
             os.rmdir(val_img_dir)
        print("Validation set formatted.")
    else:
        print("Validation set already formatted.")

download_and_format_tiny_imagenet(DATA_DIR)

Dataset already exists.
Validation set already formatted.


## 3. Dataloaders
We use Patch Size 8, so input is 64x64.

In [3]:
BATCH_SIZE = 128
NUM_WORKERS = 2

# Augmentations
train_transform = transforms.Compose([
    transforms.RandomCrop(64, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Datasets
train_ds = ImageFolder(os.path.join(DATA_DIR, 'train'), transform=train_transform)
val_ds = ImageFolder(os.path.join(DATA_DIR, 'val'), transform=val_transform)

# Loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Training classes: {len(train_ds.classes)}")
print(f"Training samples: {len(train_ds)}")

Training classes: 200
Training samples: 100000


## 4. Define Custom ViT Model (~1M Params)
We use `timm` to create a "Micro" ViT.
* **Embed Dim:** 96
* **Depth:** 8
* **Heads:** 4
* **Patch Size:** 8

In [4]:
# Original ViT model for our project
model = timm.models.vision_transformer.VisionTransformer(
    img_size=64,
    patch_size=8,
    in_chans=3,
    num_classes=200,
    embed_dim=96,    # Kept small to reduce params
    depth=8,         # 8 layers
    num_heads=4,     # 4 heads
    mlp_ratio=2.0,   # Reduced MLP ratio
    qkv_bias=True,
)

model = model.to(device)

# Count parameters
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Parameters: {param_count / 1e6:.2f} Million")

Model Parameters: 0.64 Million


## 5. Training Loop
Using AdamW and Cosine Anneling for training loop.

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

num_epochs = 30
train_accs, val_accs = [], []
train_losses, val_losses = [], []

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss/len(train_loader)})
    
    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    val_loss = val_running_loss / len(val_loader)
    
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    scheduler.step()
    
    print(f"Ep {epoch+1}: Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Val Loss: {val_loss:.4f}")

Starting training...


Epoch 1/30: 100%|██████████████████| 782/782 [01:00<00:00, 12.88it/s, loss=4.79]


Ep 1: Train Acc: 4.92% | Val Acc: 8.09% | Val Loss: 4.4463


Epoch 2/30: 100%|██████████████████| 782/782 [01:00<00:00, 12.99it/s, loss=4.23]


Ep 2: Train Acc: 11.14% | Val Acc: 13.85% | Val Loss: 4.0178


...

Ep 28: Train Acc: 50.26% | Val Acc: 39.36% | Val Loss: 2.6294


Epoch 29/30: 100%|█████████████████| 782/782 [01:00<00:00, 12.99it/s, loss=1.99]


Ep 29: Train Acc: 50.74% | Val Acc: 39.67% | Val Loss: 2.6241


Epoch 30/30: 100%|█████████████████| 782/782 [00:59<00:00, 13.09it/s, loss=1.98]


Ep 30: Train Acc: 50.64% | Val Acc: 39.62% | Val Loss: 2.6239


# Training loop (for multiple sized models)

In [5]:
# reusable training loop for multiple models (different sized ones)
def train_model(model, name="Model"):
    print(f"\n{'='*40}")
    print(f"Training {name}")
    print(f"{'='*40}")
    
    # 1. Setup Training Components
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)
    
    # 2. Loop
    num_epochs = 30
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        # Training
        for images, labels in train_loader: 
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Simple Validation (Top-1 only for speed during training)
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        scheduler.step()
        
        if (epoch + 1) % 5 == 0:
            print(f"Ep {epoch+1}/{num_epochs}: Train Loss: {running_loss/len(train_loader):.4f} | Val Acc: {val_acc:.2f}%")

    # 3. Final Detailed Evaluation (Top 1, 5, 10)
    print(f"\nRunning Final Evaluation for {name}...")
    model.eval()
    correct_1 = 0
    correct_5 = 0
    correct_10 = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            
            # Get Top-10 predictions
            _, pred = outputs.topk(10, 1, True, True)
            pred = pred.t()
            correct = pred.eq(labels.view(1, -1).expand_as(pred))
            
            # Sum up
            correct_1 += correct[:1].reshape(-1).float().sum().item()
            correct_5 += correct[:5].reshape(-1).float().sum().item()
            correct_10 += correct[:10].reshape(-1).float().sum().item()
            total += labels.size(0)
            
    print(f"-"*30)
    print(f"FINAL RESULTS: {name}")
    print(f"Top-1 Acc:  {100 * correct_1 / total:.2f}%")
    print(f"Top-5 Acc:  {100 * correct_5 / total:.2f}%")
    print(f"Top-10 Acc: {100 * correct_10 / total:.2f}%")
    print(f"-"*30)

    return model, val_acc

# Half sized model

In [6]:
# We reduce the embedding dimension to 64 to cut parameters by roughly half.

# %%
model_half = timm.models.vision_transformer.VisionTransformer(
    img_size=64,
    patch_size=8,
    in_chans=3,
    num_classes=200,
    embed_dim=64,    # Reduced from 96 -> 64
    depth=8,         # Same depth
    num_heads=4,     # 4 heads (64 is divisible by 4)
    mlp_ratio=2.0,   
    qkv_bias=True,
)

params_half = sum(p.numel() for p in model_half.parameters() if p.requires_grad)
print(f"Half-Size Model Parameters: {params_half / 1e6:.2f} Million")

# Train
model_half, accs_half = train_model(model_half, name="Half-Size ViT")

Half-Size Model Parameters: 0.30 Million

Training Half-Size ViT
Ep 5/30: Train Loss: 3.6744 | Val Acc: 20.34%
Ep 10/30: Train Loss: 3.2082 | Val Acc: 26.88%
Ep 15/30: Train Loss: 2.9222 | Val Acc: 31.02%
Ep 20/30: Train Loss: 2.7082 | Val Acc: 33.98%
Ep 25/30: Train Loss: 2.5660 | Val Acc: 35.03%
Ep 30/30: Train Loss: 2.5053 | Val Acc: 35.71%

Running Final Evaluation for Half-Size ViT...
------------------------------
FINAL RESULTS: Half-Size ViT
Top-1 Acc:  35.71%
Top-5 Acc:  62.79%
Top-10 Acc: 73.83%
------------------------------


# Double sized model (1.8M params)

In [7]:
model_double = timm.models.vision_transformer.VisionTransformer(
    img_size=64,
    patch_size=8,
    in_chans=3,
    num_classes=200,
    embed_dim=128,   # Increased from 96 -> 128
    depth=9,         # Increased from 8 -> 9
    num_heads=4,     # 4 heads (128 is divisible by 4)
    mlp_ratio=2.0,
    qkv_bias=True,
)

params_double = sum(p.numel() for p in model_double.parameters() if p.requires_grad)
print(f"Double-Size Model Parameters: {params_double / 1e6:.2f} Million")

# Train
model_double, accs_double = train_model(model_double, name="Double-Size ViT")

Double-Size Model Parameters: 1.25 Million

Training Double-Size ViT
Ep 5/30: Train Loss: 3.6136 | Val Acc: 20.58%
Ep 10/30: Train Loss: 2.9459 | Val Acc: 29.51%
Ep 15/30: Train Loss: 2.4531 | Val Acc: 35.99%
Ep 20/30: Train Loss: 2.0535 | Val Acc: 37.74%
Ep 25/30: Train Loss: 1.7516 | Val Acc: 38.58%
Ep 30/30: Train Loss: 1.6302 | Val Acc: 38.59%

Running Final Evaluation for Double-Size ViT...
------------------------------
FINAL RESULTS: Double-Size ViT
Top-1 Acc:  38.59%
Top-5 Acc:  65.20%
Top-10 Acc: 75.62%
------------------------------


## Large Image Model (LIM (PATENT PENDING) [~1 magnitude larger than baseline ~6 Million Params])


In [8]:
model_large = timm.models.vision_transformer.VisionTransformer(
    img_size=64,
    patch_size=8,
    in_chans=3,
    num_classes=200,
    embed_dim=192,   # Standard Tiny Dim
    depth=12,        # Standard Tiny Depth
    num_heads=3,     # Standard Tiny Heads (192/3 = 64 head dim)
    mlp_ratio=4.0,   # Standard Ratio
    qkv_bias=True,
)

params_large = sum(p.numel() for p in model_large.parameters() if p.requires_grad)
print(f"Large Model Parameters: {params_large / 1e6:.2f} Million")

# Train
model_large, accs_large = train_model(model_large, name="Large ViT (Tiny Config)")

Large Model Parameters: 5.43 Million

Training Large ViT (Tiny Config)
Ep 5/30: Train Loss: 3.9021 | Val Acc: 16.52%
Ep 10/30: Train Loss: 3.0499 | Val Acc: 28.12%
Ep 15/30: Train Loss: 2.2884 | Val Acc: 33.94%
Ep 20/30: Train Loss: 1.4168 | Val Acc: 35.01%
Ep 25/30: Train Loss: 0.7514 | Val Acc: 33.48%
Ep 30/30: Train Loss: 0.5374 | Val Acc: 33.20%

Running Final Evaluation for Large ViT (Tiny Config)...
------------------------------
FINAL RESULTS: Large ViT (Tiny Config)
Top-1 Acc:  33.20%
Top-5 Acc:  57.72%
Top-10 Acc: 67.81%
------------------------------


## Save to disk (so we dont have to constantly retrain)

In [9]:
# Define the models to save
models_to_save = [
    {
        "name": "vit_half_size",
        "model": model_half,
        "config": {
            'img_size': 64, 'patch_size': 8, 'in_chans': 3, 'num_classes': 200,
            'embed_dim': 64, 'depth': 8, 'num_heads': 4, 'mlp_ratio': 2.0, 'qkv_bias': True
        }
    },
    {
        "name": "vit_double_size",
        "model": model_double,
        "config": {
            'img_size': 64, 'patch_size': 8, 'in_chans': 3, 'num_classes': 200,
            'embed_dim': 128, 'depth': 9, 'num_heads': 4, 'mlp_ratio': 2.0, 'qkv_bias': True
        }
    },
    {
        "name": "vit_large_size",
        "model": model_large,
        "config": {
            'img_size': 64, 'patch_size': 8, 'in_chans': 3, 'num_classes': 200,
            'embed_dim': 192, 'depth': 12, 'num_heads': 3, 'mlp_ratio': 4.0, 'qkv_bias': True
        }
    }
]

print("Saving models to disk...")

for item in models_to_save:
    filename = f"{item['name']}.pth"
    
    checkpoint = {
        'model_state_dict': item['model'].state_dict(),
        'config': item['config']
    }
    
    torch.save(checkpoint, filename)
    print(f"  Saved: {filename:<30} | {sum(p.numel() for p in item['model'].parameters())/1e6:.2f} M Params")

print("\nAll models saved successfully.")

Saving models to disk...
  Saved: vit_half_size.pth              | 0.30 M Params
  Saved: vit_double_size.pth            | 1.25 M Params
  Saved: vit_large_size.pth             | 5.43 M Params

All models saved successfully.
